## Centerline

In [1]:
import numpy
import sys
from scipy.ndimage import distance_transform_edt
from skimage.feature import peak_local_max
from scipy.spatial import cKDTree
import vtk
from sklearn.decomposition import PCA
from scipy.interpolate import UnivariateSpline
from scipy.interpolate import BSpline
from scipy import ndimage
import trimesh
import meshio

sys.path.insert(0, "../src")  # add Folder_2 path to search list

from meshing_functions import export_centerline, getSurfaceMesh, tetra_mesh_from_stl, set_the_offset

base = "/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/meshes/"

eyeball_mesh = "/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/meshes/Segmentation_1_surf_id_1_inner.stl"
lense_mesh = "/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/meshes/re-meshed_h_1.0/id_7__h_1.0.stl"

offset = numpy.loadtxt("/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/meshes/offset.txt")
pixel_spacing = numpy.loadtxt("/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/pixel_spacing.txt")

out_folder = "extended_muscles_2/"
centerline_folder = "centerlines/"

mask_filename_base = "Segmentation_1_filtered_surf_id_" # + _1.npy

muslce_ids = [2,3,4,5]

eyeball_id = 1
lense_id = 7

mask_eyeball = numpy.load(base + mask_filename_base + str(eyeball_id) + ".npy")

print("offset = ", offset)



offset =  [33.80717894 55.61120465 26.90547222]


## 1. Lense centre & axis

In [2]:
mesh = trimesh.load(lense_mesh)
# mesh.vertices = (mesh.vertices + offset) / pixel_spacing

lense_center = mesh.vertices.mean(axis=0)

print("lense_center = ", lense_center)

X = mesh.vertices - lense_center

_, _, vh = numpy.linalg.svd(X)

axis_direction = vh[-1]  # first principal axis


projections_0 = (mesh.vertices - lense_center) @ vh[0]
lense_diameter = (projections_0.max() - projections_0.min())*pixel_spacing[0]

projections_1 = (mesh.vertices - lense_center) @ vh[-1]
lense_thickness = (projections_1.max() - projections_1.min())*pixel_spacing[0]



print("lense_diameter = ", lense_diameter)


def eyeball_axis(t):
    return lense_center + t * axis_direction


axis_points = [eyeball_axis(-20.0), eyeball_axis(20.0)]

export_centerline(axis_points, 1.0 , base + out_folder + "axis.vtp", [0,0,0])

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # 

lense_center =  [ 1.01680029  8.08340236 -2.5252567 ]
lense_diameter =  3.9438819228328748
points_np =  (2, 3)
transformed_points =  (2, 3)


## 2. Fit two planes

-  horizontal plane (left + right muslce)
- vertical plane (top + bottom muscle)

In [3]:
def load_centerline(base, centerline, cid):
    data = numpy.load(base + centerline + "centerline_id_"+str(cid)+"_parametric.npz")

    
    sx = BSpline(data["tx"], data["cx"], data["kx"])
    sy = BSpline(data["ty"], data["cy"], data["ky"])
    sz = BSpline(data["tz"], data["cz"], data["kz"])
    
    return sx, sy, sz

centerlines = [load_centerline(base, centerline_folder, cid) for cid in muslce_ids]

In [4]:
def sample_spline(sx, sy, sz, n=100):
    t_min = max(sx.t[0], sy.t[0], sz.t[0])
    t_max = min(sx.t[-1], sy.t[-1], sz.t[-1])
    
    t = numpy.linspace(t_min, t_max, n)
    
    x = sx(t)
    y = sy(t)
    z = sz(t)
    
    return numpy.vstack((x, y, z)).T  # shape (n, 3)

def sample_spline_middle(sx, sy, sz, n=100):
    t_min = max(sx.t[0], sy.t[0], sz.t[0])
    t_max = min(sx.t[-1], sy.t[-1], sz.t[-1])

    t_range = t_max - t_min
    
    t = numpy.linspace(t_min + 0.1* t_range, t_max , n)
    
    x = sx(t)
    y = sy(t)
    z = sz(t)
    
    return numpy.vstack((x, y, z)).T  # shape (n, 3)


def fit_plane(points):
    # points: (N, 3)
    centroid = points.mean(axis=0)
    
    # subtract centroid
    X = points - centroid
    
    # SVD
    _, _, vh = numpy.linalg.svd(X)
    
    normal = vh[-1]  # smallest singular vector
    
    return centroid, normal


pairs = [(0, 2), (1, 3)]

planes = []

for i, j in pairs:
    pts1 = sample_spline_middle(*centerlines[i])
    pts2 = sample_spline_middle(*centerlines[j])
    
    all_pts = numpy.vstack((pts1, pts2, lense_center[None, :]))
    
    point, normal = fit_plane(all_pts)
    
    planes.append((point, normal))


print(numpy.dot(planes[0][1], planes[1][1] ))


# remaining perpendicular direction
n3 = numpy.cross(planes[0][1], planes[1][1])
n3 = n3 / numpy.linalg.norm(n3)


0.061514159059307905


## 3. Eyeball section 

- normal vector : lense axis
- point : lense centre + dx

In [5]:
mesh_eye = trimesh.load(eyeball_mesh)

# the eyeball mesh is not translated, the lense mesh is ---> we need use the offset to get them aligned
mesh_eye.vertices = mesh_eye.vertices - offset

# visualisation only
# side = (mesh_eye.vertices - lense_center) @ axis_direction
# eps = 1e-6
# labels = numpy.zeros(len(mesh_eye.vertices))
# labels[side > eps] = 1
# labels[side < -eps] = 0

# print(labels.shape)

# meshio.write(
#     base + out_folder + "eyeball_with_labels.vtk",
#     meshio.Mesh(
#         points=mesh_eye.vertices,
#         cells=[("triangle", mesh_eye.faces)],
#         point_data={"side": labels}
#     )
# )

# section curve 

section = mesh_eye.section(
    plane_origin=lense_center - 1.5*lense_thickness*axis_direction,
    plane_normal=axis_direction
)


section_curve = []

for entity in section.entities:
    idx = entity.points  # ordered indices
    
    for i in range(len(idx) - 1):
        section_curve.append([idx[i], idx[i+1]])

meshio.write(
    base + out_folder + "section_curve.vtk",
    meshio.Mesh(
        points=section.vertices,
        cells=[("line", numpy.array(section_curve))]
    )
)

## 4. Extend centerline & merge with eyeball curvature

In [ ]:
def find_target_on_curve(section_nodes, section_curve, plane_origin, plane_normal, muscle_end_pos):
    """
    Finds the intersection between the section_curve and the plane,
    then picks the point closest to the current muscle end.
    """
    intersections = []
    
    # plane_normal must be normalized
    n = plane_normal / numpy.linalg.norm(plane_normal)
    p0 = plane_origin

    for i in range(len(section_curve)):
        # Get start and end points of the segment
        p1 = section_nodes[section_curve[i][0]]
        p2 = section_nodes[section_curve[i][1]]
        

        # Calculate signed distance of points from the plane
        # d = (p - p0) · n
        d1 = numpy.dot(p1 - p0, n)
        d2 = numpy.dot(p2 - p0, n)
        
        # If they have different signs, the segment crosses the plane
        if numpy.sign(d1) != numpy.sign(d2) and not numpy.isclose(d1, d2):
            # Line parameter t: where the intersection happens along the segment
            t = d1 / (d1 - d2)
            # Intersection point
            p_int = p1 + t * (p2 - p1)
            intersections.append(p_int)

    if not intersections:
        return None # Or handle the case where the plane doesn't hit the curve

    print("intersections = ", intersections)
    # 3. SELECT THE CORRECT POINT
    # Usually, the plane hits the 'eye ring' twice (e.g., left and right).
    # We pick the one that is closest to our existing muscle end.
    distances = [numpy.linalg.norm(pt - muscle_end_pos) for pt in intersections]
    target_point = intersections[numpy.argmin(distances)]
    
    return target_point


def extend_centerline_to_eye(sx, sy, sz, eyeball_mesh, target_point, n_ext=20):
    # --- Step 1: Get end state of existing centerline ---
    # We evaluate at t=1.0
    p_end = numpy.array([sx(1.0), sy(1.0), sz(1.0)])
    # Approximate tangent at the end
    dt = 0.01
    p_prev = numpy.array([sx(1.0-dt), sy(1.0-dt), sz(1.0-dt)])

    print("p_end = ", p_end)
    print("p_prev = ", p_prev)
    print()

    tangent = (p_end - p_prev)
    tangent /= numpy.linalg.norm(tangent)

    # --- Step 2: Intersection with Eyeball ---
    # Create a ray starting slightly before the end, pointing along the tangent
    ray_origin = p_end.reshape(1, 3)
    ray_direction = tangent.reshape(1, 3)
    
    locations, index_ray, index_tri = eyeball_mesh.ray.intersects_location(
        ray_origins=ray_origin, ray_directions=ray_direction)

    if len(locations) == 0:
        print("no intersection")
        # Fallback: find closest point on mesh if ray misses
        p_intersect = eyeball_mesh.nearest.on_surface(numpy.reshape(p_end + tangent * 5.0,(1,3)))[0]
        print("closest point = ", p_intersect)
    else:
        # Get the first intersection point
        p_intersect = locations[0]
        print("p_intersect = ", p_intersect)

    # --- Step 3: Wrap along the surface ---
    # To follow the eye curvature, we interpolate between p_intersect 
    # and the target_point, then normalize the result to the eye's radius.
    eye_center = eyeball_mesh.center_mass
    
    # Generate points along a "great circle" path
    ext_points = []
    # Include a few points from p_end to p_intersect to bridge the gap
    bridge = numpy.linspace(p_end, p_intersect, 5)
    ext_points.extend(bridge[1:])

    # Generate points from intersection to the curve target
    for alpha in numpy.linspace(0, 1, n_ext):
        # Linear interpolation in 3D
        p_lin = p_intersect + alpha * (target_point - p_intersect)
        
        # Project back to sphere surface: 
        # Move vector from center to p_lin, normalize, and scale by eye radius
        vec = p_lin - eye_center
        dist = numpy.linalg.norm(p_end - eye_center) # Use muscle distance as radius
        p_wrapped = eye_center + (vec / numpy.linalg.norm(vec)) * dist
        ext_points.append(p_wrapped)

    return numpy.array(ext_points)


i = 1
sx, sy, sz = centerlines[i]

section_nodes_image_space = (section.vertices + offset) / pixel_spacing 

muscle_end_pos = numpy.array([sx(1.0), sy(1.0), sz(1.0)])

target_on_curve = find_target_on_curve(section_nodes_image_space, section_curve , planes[0][0], planes[0][1], muscle_end_pos)

print("target_on_curve = ", target_on_curve)

target_on_curve_phys = target_on_curve * pixel_spacing - offset
print("target_on_curve_phys = ", target_on_curve_phys)
print()


extension_path = extend_centerline_to_eye(sx, sy, sz, mesh_eye, target_on_curve)



intersections =  [TrackedArray([ 92.00050786, 123.02309155,  56.90291654]), TrackedArray([ 44.16228945, 123.83638604,  54.56903425])]
target_on_curve =  [ 44.16228945 123.83638604  54.56903425]
target_on_curve_phys =  [-11.72603421   6.30698836   0.37904491]

p_end =  [ -6.67368384 -16.24362497  10.35354711]
p_prev =  [ -6.67579312 -16.24795783  10.35219795]

no intersection
closest point =  [[-3.26418279 -8.7906132   9.16208688]]


## Visualisation of horizontal and vertical plane

In [7]:
# section = mesh_eye.section(
#     plane_origin= lense_center,
#     plane_normal= planes[0][1]
# )


# section_curve = []

# for entity in section.entities:
#     idx = entity.points  # ordered indices
    
#     for i in range(len(idx) - 1):
#         section_curve.append([idx[i], idx[i+1]])

# meshio.write(
#     base + out_folder + "horizontal_plane_section_curve.vtk",
#     meshio.Mesh(
#         points=section.vertices,
#         cells=[("line", numpy.array(section_curve))]
#     )
# )

# section = mesh_eye.section(
#     plane_origin= lense_center,
#     plane_normal= planes[1][1]
# )


# section_curve = []

# for entity in section.entities:
#     idx = entity.points  # ordered indices
    
#     for i in range(len(idx) - 1):
#         section_curve.append([idx[i], idx[i+1]])

# meshio.write(
#     base + out_folder + "vertical_plane_section_curve.vtk",
#     meshio.Mesh(
#         points=section.vertices,
#         cells=[("line", numpy.array(section_curve))]
#     )
# )

In [8]:

export_centerline(extension_path.squeeze(1), pixel_spacing[0] , base + out_folder + "extention_id_"+str(muslce_ids[i])+".vtp", offset)


points_np =  (24, 3)
transformed_points =  (24, 3)
